# Notebook 06 — Knowledge Graph + Graph-Aware Context Assembly

**Goal:** connect the reranked chunks to the document structure and multimodal evidence so the final RAG context is not just a flat list of chunks.

### Pipeline
```text
Notebook 05
Top 5 Reranked Chunks
        │
        ▼
Knowledge Graph
        │
        ├── Document → Section
        ├── Section → Chunk
        ├── Chunk → Element
        ├── Chunk → Figure / Table / Formula
        └── Chunk ↔ Related Chunk
        │
        ▼
Graph-Aware Context Expansion
        │
        ▼
Context Candidates
        │
        ▼
Notebook 07 / RAG Generation
```

This notebook keeps the graph intentionally simple and traceable. It does **not** build a large ontology.


In [1]:
# 01 — Imports and configuration

import json
from pathlib import Path
from collections import defaultdict

import networkx as nx

DATA_DIR = Path("../data")
VECTOR_STORE_DIR = DATA_DIR / "vector_store"

METADATA_PATH = VECTOR_STORE_DIR / "quant_research_metadata.json"
GRAPH_PATH = VECTOR_STORE_DIR / "quant_research_knowledge_graph.graphml"
GRAPH_DATA_PATH = VECTOR_STORE_DIR / "quant_research_knowledge_graph.json"

FINAL_TOP_K = 5
MAX_CONTEXT_CHUNKS = 10
RELATED_CHUNKS_PER_RESULT = 2

print("Metadata:", METADATA_PATH)
print("Graph:", GRAPH_PATH)


Metadata: ..\data\vector_store\quant_research_metadata.json
Graph: ..\data\vector_store\quant_research_knowledge_graph.graphml


In [2]:
# 02 — Load chunk metadata

assert METADATA_PATH.exists(), f"Missing metadata: {METADATA_PATH}"

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

metadata_by_id = {m["chunk_id"]: m for m in metadata}

print("Metadata records:", len(metadata))
print("Unique chunk IDs:", len(metadata_by_id))

assert len(metadata) == len(metadata_by_id)


Metadata records: 1141
Unique chunk IDs: 1141


In [3]:
# 03 — Build the structural Knowledge Graph

G = nx.MultiDiGraph()

for record in metadata:
    chunk_id = record["chunk_id"]
    document_id = record.get("document_id", record.get("source_file", "unknown_document"))
    source_file = record.get("source_file", "unknown_source")
    section_path = record.get("section_path") or []

    # Document node
    document_node = f"document::{document_id}"
    G.add_node(
        document_node,
        node_type="document",
        document_id=document_id,
        source_file=source_file,
    )

    # Chunk node
    chunk_node = f"chunk::{chunk_id}"
    G.add_node(
        chunk_node,
        node_type="chunk",
        chunk_id=chunk_id,
        source_file=source_file,
        pages=json.dumps(record.get("pages", [])),
        content_types=json.dumps(record.get("content_types", [])),
    )

    G.add_edge(document_node, chunk_node, relation="contains_chunk")

    # Section hierarchy + section-to-chunk relation
    parent_section_node = document_node

    for level, section_name in enumerate(section_path):
        section_key = f"{document_id}::section::{level}::{section_name}"
        section_node = f"section::{section_key}"

        if section_node not in G:
            G.add_node(
                section_node,
                node_type="section",
                document_id=document_id,
                section_name=section_name,
                level=level,
            )

        if not G.has_edge(parent_section_node, section_node):
            G.add_edge(
                parent_section_node,
                section_node,
                relation="contains_section",
            )

        parent_section_node = section_node

    if section_path:
        G.add_edge(
            parent_section_node,
            chunk_node,
            relation="contains_chunk",
        )

    # Explicit element/evidence links
    for element_id in record.get("element_ids", []):
        element_node = f"element::{element_id}"
        G.add_node(
            element_node,
            node_type="element",
            element_id=element_id,
        )
        G.add_edge(chunk_node, element_node, relation="contains_element")

    for figure_id in record.get("figure_ids", []):
        node = f"figure::{figure_id}"
        G.add_node(node, node_type="figure", figure_id=figure_id)
        G.add_edge(chunk_node, node, relation="references_figure")

    for table_id in record.get("table_ids", []):
        node = f"table::{table_id}"
        G.add_node(node, node_type="table", table_id=table_id)
        G.add_edge(chunk_node, node, relation="references_table")

    for formula_id in record.get("formula_ids", []):
        node = f"formula::{formula_id}"
        G.add_node(node, node_type="formula", formula_id=formula_id)
        G.add_edge(chunk_node, node, relation="references_formula")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


Nodes: 3426
Edges: 4561


In [4]:
# 04 — Add sequential chunk relationships

# Chunks are linked when they belong to the same source document.
# This provides lightweight local context expansion.

chunks_by_document = defaultdict(list)

for record in metadata:
    chunks_by_document[
        record.get("document_id", record.get("source_file", "unknown_document"))
    ].append(record)

for document_id, records in chunks_by_document.items():
    records.sort(key=lambda x: x.get("chunk_index", 0))

    for current, nxt in zip(records, records[1:]):
        current_node = f"chunk::{current['chunk_id']}"
        next_node = f"chunk::{nxt['chunk_id']}"

        G.add_edge(
            current_node,
            next_node,
            relation="next_chunk",
        )

        G.add_edge(
            next_node,
            current_node,
            relation="previous_chunk",
        )

print("Graph after sequential links:")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


Graph after sequential links:
Nodes: 3426
Edges: 6821


In [5]:
# 05 — Add same-section relationships

chunks_by_section = defaultdict(list)

for record in metadata:
    section_key = (
        record.get("document_id", record.get("source_file", "unknown_document")),
        tuple(record.get("section_path") or []),
    )

    chunks_by_section[section_key].append(record)

for _, records in chunks_by_section.items():
    records.sort(key=lambda x: x.get("chunk_index", 0))

    for i, record in enumerate(records):
        current_node = f"chunk::{record['chunk_id']}"

        for neighbor in records[max(0, i - RELATED_CHUNKS_PER_RESULT): i]:
            neighbor_node = f"chunk::{neighbor['chunk_id']}"

            G.add_edge(
                current_node,
                neighbor_node,
                relation="same_section",
            )

            G.add_edge(
                neighbor_node,
                current_node,
                relation="same_section",
            )

print("Graph with same-section relationships:")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


Graph with same-section relationships:
Nodes: 3426
Edges: 10111


In [6]:
# 06 — Save graph

# GraphML is convenient for visualization/interoperability.
# JSON preserves richer node/edge attributes.

nx.write_graphml(G, GRAPH_PATH)

graph_payload = {
    "nodes": [
        {"id": node, **attrs}
        for node, attrs in G.nodes(data=True)
    ],
    "edges": [
        {"source": source, "target": target, **attrs}
        for source, target, attrs in G.edges(data=True)
    ],
}

with open(GRAPH_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(graph_payload, f, indent=2, default=str)

print("GraphML saved:", GRAPH_PATH)
print("Graph JSON saved:", GRAPH_DATA_PATH)


GraphML saved: ..\data\vector_store\quant_research_knowledge_graph.graphml
Graph JSON saved: ..\data\vector_store\quant_research_knowledge_graph.json


In [7]:
# 07 — Graph QA

node_types = defaultdict(int)

for _, attrs in G.nodes(data=True):
    node_types[attrs.get("node_type", "unknown")] += 1

relations = defaultdict(int)

for _, _, attrs in G.edges(data=True):
    relations[attrs.get("relation", "unknown")] += 1

print("Node types:")
for key, value in sorted(node_types.items()):
    print(f"  {key}: {value}")

print("\nEdge relations:")
for key, value in sorted(relations.items()):
    print(f"  {key}: {value}")

assert G.number_of_nodes() > 0
assert G.number_of_edges() > 0
assert node_types["chunk"] == len(metadata)

print("\nKnowledge Graph QA: PASS")


Node types:
  chunk: 1141
  document: 11
  element: 1582
  figure: 77
  formula: 301
  section: 246
  table: 68

Edge relations:
  contains_chunk: 2282
  contains_element: 1587
  contains_section: 246
  next_chunk: 1130
  previous_chunk: 1130
  references_figure: 77
  references_formula: 301
  references_table: 68
  same_section: 3290

Knowledge Graph QA: PASS


In [8]:
# 08 — Load final reranked results from Notebook 05

# Notebook 05 normally keeps `reranked_results` in memory.
# This cell also supports a standalone run by creating a test retrieval
# from the stored metadata if the variable is not available.

if "reranked_results" in globals():
    final_results = reranked_results[:FINAL_TOP_K]
else:
    # Standalone fallback: use the first available chunks only for graph testing.
    # In the full pipeline, Notebook 05 results should be supplied instead.
    final_results = [
        {
            "chunk_id": record["chunk_id"],
            "rerank_rank": i + 1,
            "rerank_score": 0.0,
        }
        for i, record in enumerate(metadata[:FINAL_TOP_K])
    ]

print("Input reranked chunks:", len(final_results))


Input reranked chunks: 5


In [10]:
# 09 — Graph-aware context expansion

def graph_neighbors_for_chunk(chunk_id):
    node = f"chunk::{chunk_id}"

    if node not in G:
        return []

    neighbors = []

    for neighbor, edge_data in G[node].items():
        relations = {
            attrs.get("relation")
            for attrs in edge_data.values()
        }

        if neighbor.startswith("chunk::"):
            neighbors.append({
                "chunk_id": neighbor.replace("chunk::", ""),
                "relations": sorted(r for r in relations if r),
            })

    return neighbors


def graph_aware_context(reranked_results, max_chunks=MAX_CONTEXT_CHUNKS):
    selected = []
    selected_ids = set()

    # First preserve the reranked order.
    for result in reranked_results:
        chunk_id = result["chunk_id"]

        if chunk_id not in selected_ids:
            selected.append({
                "chunk_id": chunk_id,
                "selection_reason": "reranked",
                "rerank_rank": result.get("rerank_rank"),
                "rerank_score": result.get("rerank_score"),
            })
            selected_ids.add(chunk_id)

        if len(selected) >= max_chunks:
            return selected

    # Then expand locally through graph relationships.
    for result in reranked_results:
        chunk_id = result["chunk_id"]

        for neighbor in graph_neighbors_for_chunk(chunk_id):
            neighbor_id = neighbor["chunk_id"]

            if neighbor_id in selected_ids:
                continue

            selected.append({
                "chunk_id": neighbor_id,
                "selection_reason": "graph_neighbor",
                "related_to": chunk_id,
                "relations": neighbor["relations"],
            })

            selected_ids.add(neighbor_id)

            if len(selected) >= max_chunks:
                return selected

    return selected


In [11]:
# 10 — Build graph-aware context

context_candidates = graph_aware_context(
    final_results,
    max_chunks=MAX_CONTEXT_CHUNKS,
)

print("Final reranked chunks:", len(final_results))
print("Graph-aware context candidates:", len(context_candidates))

for item in context_candidates:
    print(
        item["chunk_id"],
        "|",
        item["selection_reason"],
        "|",
        item.get("relations", ""),
    )


Final reranked chunks: 5
Graph-aware context candidates: 7
chunk_634b3e042dffad81 | reranked | 
chunk_e9dbc9860c200c4a | reranked | 
chunk_f0d41995c19a42f8 | reranked | 
chunk_09712835319a6dd6 | reranked | 
chunk_d9600aaaec5dcae2 | reranked | 
chunk_f1c8dbdb6d16fb1b | graph_neighbor | ['same_section']
chunk_424b6cfa3bb0365e | graph_neighbor | ['same_section']


In [12]:
# 11 — Attach full metadata for context assembly

context_records = []

for item in context_candidates:
    chunk_id = item["chunk_id"]

    if chunk_id not in metadata_by_id:
        continue

    record = metadata_by_id[chunk_id].copy()
    record["selection_reason"] = item["selection_reason"]

    if "related_to" in item:
        record["related_to"] = item["related_to"]

    if "relations" in item:
        record["graph_relations"] = item["relations"]

    context_records.append(record)

print("Context records:", len(context_records))

assert all(
    record["chunk_id"] in metadata_by_id
    for record in context_records
)

print("Context assembly QA: PASS")


Context records: 7
Context assembly QA: PASS


In [13]:
# 12 — Preview graph-aware context

for i, record in enumerate(context_records, start=1):
    print("=" * 100)
    print("Context rank:", i)
    print("Chunk:", record["chunk_id"])
    print("Selection:", record.get("selection_reason"))
    print("Source:", record.get("source_file"))
    print("Pages:", record.get("pages"))
    print("Section:", record.get("section_path"))
    print("Content types:", record.get("content_types"))
    print("Graph relations:", record.get("graph_relations", []))
    print("Text:", (record.get("retrieval_text") or record.get("text") or "")[:500])


Context rank: 1
Chunk: chunk_634b3e042dffad81
Selection: reranked
Source: ssrn-5231379.pdf
Pages: [1]
Section: ['QUANTS AND MARKET EFFICIENCY']
Content types: ['text']
Graph relations: []
Text: NIHAD ALIYEV AND XINYI DENG University of Technology Sydney Abstract. We study the impact of quantitative trading strategies (quants) on price efficiency by developing a model that incorporates two defining features of quants: superior information processing capacity and limited strategic awareness. These forces have opposing effects on trading aggressiveness-better information pro- cessing increases it, while limited strategic awareness decreases it. As a result, an increase in quant participat
Context rank: 2
Chunk: chunk_e9dbc9860c200c4a
Selection: reranked
Source: ssrn-5231379.pdf
Pages: [2]
Section: ['1. Introduction']
Content types: ['text']
Graph relations: []
Text: Advances in artificial intelligence (AI), along with the availability of big data and computing power, have changed how inve

In [14]:
# 13 — Persist context candidates for Notebook 07

CONTEXT_PATH = VECTOR_STORE_DIR / "graph_aware_context.json"

with open(CONTEXT_PATH, "w", encoding="utf-8") as f:
    json.dump(context_records, f, indent=2, default=str)

print("Saved graph-aware context:", CONTEXT_PATH)


Saved graph-aware context: ..\data\vector_store\graph_aware_context.json


## Notebook 06 → Notebook 07 contract

Notebook 07 will receive **graph-aware context**, not just independent retrieved chunks.

```text
Top 5 Reranked Chunks
        ↓
Knowledge Graph
        ↓
Local structural expansion
        ↓
Context Candidates
        ↓
Context Assembly
        ↓
RAG Generation
        ↓
Grounded Answer
        ↓
Citations + Traceability
```

Each context record retains:

- `chunk_id`
- `source_file`
- `section_path`
- `pages`
- `element_ids`
- `figure_ids`
- `table_ids`
- `formula_ids`
- `image_paths`
- `evidence`
- graph selection/relation metadata

**Important:** the graph is used to improve context selection. It does not replace dense retrieval, BM25, RRF, or Cross-Encoder reranking.
